# 03 OTALA

In [88]:
import pandas as pd

clean = pd.read_parquet("../data/processed/df_clean.parquet")
train = pd.read_parquet("../data/processed/train.parquet")
test  = pd.read_parquet("../data/processed/test.parquet")

In [89]:
oil0 = (clean["Oil_level"] == 0)

# 1) group by month
print(oil0.groupby(clean.index.month).mean().round(3))

# 2) compare leak and normal
LEAKS = [("2020-04-18 00:00","2020-04-18 23:59"), ("2020-05-29 23:30","2020-05-30 06:00"),
         ("2020-06-05 10:00","2020-06-07 14:30"), ("2020-07-15 14:30","2020-07-15 19:00")]
leak = pd.Series(False, index=clean.index)
for s, e in LEAKS:
    leak |= (clean.index >= s) & (clean.index <= e)
print("Leck:", oil0[leak].mean().round(3), " Normal:", oil0[~leak].mean().round(3))

timestamp
2    0.000
3    0.004
4    0.036
5    0.012
6    0.021
7    0.037
8    0.553
9    0.000
Name: Oil_level, dtype: float64
Leck: 0.001  Normal: 0.098


Oil_level: Nullwerte treten hauptsächlich im August auf (55 %), während der Lecks fast nie (0,1 %) -> für die Leckerkennung nicht relevant

## Auswahl der binären Signale für OTALA

OTALA verarbeitet nur binäre Signale (0/1), da das Modell aus Zustandswechseln (Ereignissen) gelernt wird. Von den acht binären Signalen in MetroPT-3 wurden vier ausgewählt.

| Signal | Verwendet | Begründung |
|---|---|---|
| **DV_eletric** | ✅ ja | zeigt an, ob der Kompressor pumpt |
| **COMP** | ✅ ja | meist gegensätzlich zu DV_eletric (φ = −0,96), zeigt aber zwei zusätzliche Zustände: beide 0 (Kompressor aus, 0,7 %) und beide 1 (Übergang am Ende des Pumpens, 0,4 %, relevant für H6) |
| **Towers** | ✅ ja | Trocknertürme wechseln während des Pumpens |
| **LPS** | ✅ ja | Niederdrucksignal, selten (ca. 0,3 %), aber bei Leck 4 zu 33 % aktiv |
| MPG | ❌ nein | nahezu identisch mit COMP (φ = 0,98, in 99,6 % gleich) |
| Pressure_switch | ❌ nein | zu 99,1 % = 1, kein erkennbarer Zusammenhang mit Pumpen oder Lecks |
| Oil_level | ❌ nein| Nullwerte treten hauptsächlich im August auf (55 %), während der Lecks fast nie (0.1%) |



In [90]:
import pandas as pd
import graphviz
import textwrap


In [91]:
SIG = ["DV_eletric", "COMP", "Towers", "LPS"]
NAMES = {"DV_eletric": "DV", "COMP": "COMP", "Towers": "T", "LPS": "LPS"}
EVENT_TEXT = {"DV↑": "Kompressor beginnt zu pumpen", "DV↓": "Kompressor hört auf zu pumpen",
              "COMP↑": "Luftansaugung gestoppt", "COMP↓": "Luftansaugung beginnt",
              "T↑": "Wechsel auf Turm 1", "T↓": "Wechsel auf Turm 0",
              "LPS↑": "Druck fällt unter Grenzwert", "LPS↓": "Druck wieder normal"}

def explain_event(ev):
    return ", ".join(EVENT_TEXT.get(x, x) for x in ev.split())

def vec_of(row):
    return tuple(int(round(x)) for x in row)

def event_name(old, new):
    parts = []
    for s, a, b in zip(SIG, old, new):
        if a != b:
            parts.append(NAMES[s] + ("↑" if b == 1 else "↓"))
    return " ".join(parts)

def to_tables(states, edges, data, min_count=5):
    st = pd.DataFrame([(n, v) for v, n in states.items()], columns=["Zustand", "Vektor"])
    v_all = data[SIG].round().astype(int).apply(tuple, axis=1)
    share = v_all.value_counts(normalize=True)
    st["Anteil_%"] = st["Vektor"].map(share).fillna(0).mul(100).round(2)
    st = st.sort_values("Anteil_%", ascending=False)

    rows = [(a, e, b, len(v), min(v), max(v)) for (a, e, b), v in edges.items()]
    tab = pd.DataFrame(rows, columns=["von", "Ereignis", "nach", "n", "t_min", "t_max"])
    tab["selten"] = tab["n"] < min_count
    return st, tab.sort_values("n", ascending=False).reset_index(drop=True)

def draw(st, tab, file="otala_vector", min_count=5):
    g = graphviz.Digraph(graph_attr=dict(rankdir="LR", fontname="Times"),
                         node_attr=dict(shape="circle", fontname="Times", fontsize="11"),
                         edge_attr=dict(fontname="Times", fontsize="9"))
    for r in st.itertuples():
        vec_str = str(r.Vektor).replace(" ", "")
        lab = r.Zustand + "\n" + vec_str
        g.node(r.Zustand, lab, shape="doublecircle" if r.Vektor == (0, 1, 1, 0) else "circle")
    for r in tab[tab.n >= min_count].itertuples():
        col = "blue" if ("DV↑" in r.Ereignis and "COMP↓" in r.Ereignis) else "black"
        g.edge(r.von, r.nach, color=col, fontcolor=col,
               label=f"{r.Ereignis}\n[{r.t_min:.1f}–{r.t_max:.1f}] n={r.n}")
    g.node("legend", shape="note", fontsize="9", label=(
        "Legende\\l"
        "DV = Kompressor pumpt (1)\\l"
        "COMP = keine Luftansaugung (1)\\l"
        "T = aktiver Trocknerturm (0/1)\\l"
        "LPS = Druck zu niedrig (1)\\l"
        "↑ = 0→1   ↓ = 1→0\\l"
        "Vektor = (DV, COMP, T, LPS)\\l"))
    g.render(file, format="png", cleanup=True)
    return g

def draw_compare(states, tab_train, test, file="otala_test", min_count=5):
    states_te, edges_te = otala_learn(test, states.copy())  # Kopie -> Training-Dict bleibt unverändert
    st_te, tab_te = to_tables(states_te, edges_te, test)
    known = set(map(tuple, tab_train.loc[tab_train.n >= min_count, ["von", "Ereignis", "nach"]].values))
    seen = set(map(tuple, tab_te[["von", "Ereignis", "nach"]].values))

    g = graphviz.Digraph(graph_attr=dict(rankdir="LR", fontname="Times",
                         label="schwarz = wie im Training   rot gestrichelt = neu im Test   grau = nur im Training",
                         fontsize="10"),
                         node_attr=dict(shape="circle", fontname="Times", fontsize="11"),
                         edge_attr=dict(fontname="Times", fontsize="9"))
    for r in st_te.itertuples():
        vec_str = str(r.Vektor).replace(" ", "")
        lab = r.Zustand + "\n" + vec_str
        g.node(r.Zustand, lab, shape="doublecircle" if r.Vektor == (0, 1, 1, 0) else "circle")
    for r in tab_te[tab_te.n >= min_count].itertuples():
        new = (r.von, r.Ereignis, r.nach) not in known
        g.edge(r.von, r.nach, color="red" if new else "black", fontcolor="red" if new else "black",
               style="dashed" if new else "solid",
               label=f"{r.Ereignis}\n[{r.t_min:.1f}–{r.t_max:.1f}] n={r.n}" + ("\nNEU" if new else ""))
    for (a, e, b) in known - seen:
        g.edge(a, b, label=e, color="gray", fontcolor="gray")
    g.render(file, format="png", cleanup=True)
    tab_te["neu"] = [(a, e, b) not in known for a, e, b in tab_te[["von", "Ereignis", "nach"]].values]
    return st_te, tab_te


In [92]:
def otala_learn(data, states=None):
    states = dict(states) if states else {}
    edges = {}                                # (von, Ereignis, nach) -> Liste der Δt
    for seg, s in data.groupby("segment"):
        X = s[SIG].to_numpy()
        T = s.index

        vec = vec_of(X[0])
        cur = states.setdefault(vec, f"S{len(states)}")
        t_prev = T[0]

        for i in range(1, len(X)):
            new = vec_of(X[i])
            if new == vec:
                continue
            nxt = states.setdefault(new, f"S{len(states)}")
            dt = (T[i] - t_prev).total_seconds() / 60
            edges.setdefault((cur, event_name(vec, new), nxt), []).append(dt)
            vec, cur, t_prev = new, nxt, T[i]
    return states, edges

def pump_limit_from(tab, data, q=0.995):
    d = []
    for seg, s in data.groupby("segment"):
        on = s["DV_eletric"].round().astype(int)
        grp = (on.diff() != 0).cumsum()
        for g, part in s[on == 1].groupby(grp[on == 1]):
            if part.index[0] == s.index[0]:
                continue
            d.append((part.index[-1] - part.index[0]).total_seconds() / 60)
    return pd.Series(d).quantile(q)

In [93]:
def anoda(data, states, tab, pump_limit, min_count=5):
    """Prüft neue Daten gegen das gelernte Modell."""
    model = tab[tab.n >= min_count].set_index(["von", "Ereignis"])
    max_wait = model.groupby(level=0)["t_max"].max()
    res = []

    for seg, s in data.groupby("segment"):
        X = s[SIG].to_numpy()
        T = s.index
        vec = vec_of(X[0])
        cur = states.get(vec)
        t_prev = T[0]
        pump_start = T[0] if vec[0] == 1 else None
        pump_flag = False

        for i in range(1, len(X)):
            new = vec_of(X[i])
            # Pumpdauer-Timeout
            if pump_start is not None and not pump_flag and \
                    (T[i] - pump_start).total_seconds() / 60 > pump_limit:
                res.append((T[i], cur, "-", "Timeout (Pumpdauer)"))
                pump_flag = True
            if new == vec:
                continue
            ev = event_name(vec, new)
            dt = (T[i] - t_prev).total_seconds() / 60

            if cur is not None:   # unbekannter Startzustand -> keine Prüfung
                if cur in max_wait.index and dt > max_wait[cur]:
                    res.append((T[i], cur, ev, "Timeout"))
                if (cur, ev) not in model.index:
                    res.append((T[i], cur, ev, "Unknown Event"))
                else:
                    r = model.loc[(cur, ev)]
                    if not (r.t_min <= dt <= r.t_max):
                        res.append((T[i], cur, ev, "Wrong Timing"))

            # Pumpuhr starten / stoppen
            if vec[0] == 0 and new[0] == 1:
                pump_start, pump_flag = T[i], False
            if vec[0] == 1 and new[0] == 0:
                pump_start = None

            cur = states.get(new)
            vec, t_prev = new, T[i]
    return pd.DataFrame(res, columns=["time", "Zustand", "Ereignis", "anoda"])

In [94]:
states, edges = otala_learn(train)
st, tab = to_tables(states, edges, train)
limit = pump_limit_from(tab, train)
print(limit)


10.113333333333381


In [95]:
print(st.to_string(index=False))
print(tab.round(2).to_string())
_ = draw(st, tab,"otala_train")

Zustand       Vektor  Anteil_%
     S0 (0, 1, 1, 0)     87.78
     S1 (1, 0, 0, 0)      6.05
     S2 (1, 0, 1, 0)      6.04
     S4 (1, 0, 0, 1)      0.06
     S5 (1, 0, 1, 1)      0.06
     S3 (1, 1, 1, 0)      0.00
   von           Ereignis nach     n  t_min  t_max  selten
0   S2                 T↓   S1  5191   0.15   1.17   False
1   S1                 T↑   S2  5105   0.15   1.17   False
2   S1       DV↓ COMP↑ T↑   S0  1869   0.15   1.17   False
3   S0       DV↑ COMP↓ T↓   S1  1767   0.15  44.77   False
4   S0          DV↑ COMP↓   S2  1708   0.15  45.58   False
5   S2          DV↓ COMP↑   S0  1640   0.15   1.17   False
6   S5                 T↓   S4    58   0.15   1.15   False
7   S4                 T↑   S5    55   0.17   1.00   False
8   S5               LPS↓   S2    14   0.17   0.82   False
9   S4               LPS↓   S1    11   0.15   0.83   False
10  S3              COMP↓   S2     6   0.17   0.17   False
11  S2              COMP↑   S3     5   0.17   1.00   False
12  S0          

In [96]:
res = anoda(test, states, tab, limit)
print(res["anoda"].value_counts())
fz = test["frozen"]
res["Typ"] = ["Daten-/Sensoranomalie" if fz.get(t, False) else "Prozessanomalie"
              for t in res["time"]]
print(pd.crosstab(res["anoda"], res["Typ"]))

anoda
Unknown Event          9051
Timeout                1811
Timeout (Pumpdauer)      41
Wrong Timing             15
Name: count, dtype: int64
Typ                  Daten-/Sensoranomalie  Prozessanomalie
anoda                                                      
Timeout                               1809                2
Timeout (Pumpdauer)                      0               41
Unknown Event                         8980               71
Wrong Timing                             0               15


In [97]:
st_te, tab_te = draw_compare(states, tab, test,file="otala_test_alle")
print(tab_te[tab_te.neu & (tab_te.n >= 5)].round(2).to_string())

   von      Ereignis nach     n  t_min  t_max  selten   neu
2   S0      COMP↓ T↓   S6  4364   0.18  32.40   False  True
3   S6      COMP↑ T↑   S0  4362   0.18   0.82   False  True
8   S6        DV↑ T↑   S2  3226   0.18   0.82   False  True
9   S2        DV↓ T↓   S6  3226   0.17   5.03   False  True
10  S3  DV↓ COMP↓ T↓   S6  1388   0.18   2.62   False  True
11  S6  DV↑ COMP↑ T↑   S3  1387   0.20   0.62   False  True
16  S0           DV↑   S3    11   0.33  30.40   False  True
17  S3           DV↓   S0    11   0.15   2.63   False  True
19  S5       T↓ LPS↓   S1     9   0.33   1.00   False  True
21  S3      COMP↓ T↓   S1     8   0.15   0.17   False  True
22  S1      COMP↑ T↑   S3     7   0.33   1.00   False  True
23  S1          LPS↑   S4     6   0.17   0.83   False  True
24  S2          LPS↑   S5     5   0.17   0.65   False  True
25  S4       T↑ LPS↓   S2     5   0.98   1.00   False  True
26  S2       T↓ LPS↑   S4     5   0.83   1.00   False  True


In [98]:
test_proz = test[~test["frozen"]].copy()
test_proz["segment"] = (test_proz.index.to_series().diff() > pd.Timedelta("1min")).cumsum().values
st_te, tab_te = draw_compare(states, tab, test_proz,file="otala_test_prozess")

In [99]:
LEAKS = [("2020-04-18 00:00", "2020-04-18 23:59"), ("2020-05-29 23:30", "2020-05-30 06:00"),
             ("2020-06-05 10:00", "2020-06-07 14:30"), ("2020-07-15 14:30", "2020-07-15 19:00")]
for i, (s, e) in enumerate(LEAKS, 1):             # Labels NUR zur Bewertung
    s, e = pd.Timestamp(s), pd.Timestamp(e)
    w = res[res.time.between(s - pd.Timedelta("24h"), e)]
    p = w[w.anoda == "Timeout (Pumpdauer)"].time.min()
    f = test.loc[s - pd.Timedelta("24h"):e, "frozen"] #[True, False...]
    f = f[f] #keep True, remove False
    print(f"Leck {i}: {w.anoda.value_counts().to_dict()} | erster Pumpdauer-Alarm: {p} ")
    if len(f) > 0:
        print("   frozen:", len(f), "Zeilen, von", f.index.min(), "bis", f.index.max())
    else:
        print("   frozen: keine")


Leck 1: {'Unknown Event': 770, 'Timeout (Pumpdauer)': 1} | erster Pumpdauer-Alarm: 2020-04-18 00:34:13 
   frozen: 4470 Zeilen, von 2020-04-17 09:20:33 bis 2020-04-18 00:18:07
Leck 2: {'Unknown Event': 2, 'Timeout (Pumpdauer)': 1} | erster Pumpdauer-Alarm: 2020-05-29 23:25:10 
   frozen: keine
Leck 3: {'Timeout (Pumpdauer)': 3, 'Unknown Event': 1} | erster Pumpdauer-Alarm: 2020-06-05 09:58:45 
   frozen: keine
Leck 4: {'Timeout (Pumpdauer)': 2, 'Unknown Event': 2} | erster Pumpdauer-Alarm: 2020-07-15 14:35:37 
   frozen: keine
